# Inductive vs. Transductive Learning and Online Regret vs. Batch IID (Topic 12)

### Historical Lineage & Theoretical Foundations
Machine learning paradigms bifurcate along observation horizons and structural assumptions:
- **Transductive Learning (Vapnik 1998; Zhou et al. 2003):** Direct graph propagation $F^* = (I - \alpha S)^{-1}Y$ over known manifolds without learning general inductive functional mappings.
- **Inductive Learning (Vapnik 1982; Hastie et al. 2009):** Fits an out-of-sample functional hypothesis $f: \mathcal{X} \to \mathcal{Y}$ mapping unseen test points in $\mathcal{O}(D)$ time.
- **Online Convex Optimization & FTRL (Hazan 2016; McMahan et al. 2013):** Minimizes regret $\mathcal{R}_T = \sum f_t(w_t) - \min_w \sum f_t(w) \le \mathcal{O}(\sqrt{T})$ under arbitrary non-IID adversary streams with exact $L_1$ coordinate sparsity.
- **Sequential Change Detection (Page 1954; Lorden 1971):** Cumulative sum (CUSUM) and Page-Hinkley stopping rules detecting non-stationary distribution shifts.

### Decision Boundary: When NOT to Use
> [!WARNING]
> Do NOT use Transductive inference for high-throughput low-latency serving (requires $\mathcal{O}(N^3)$ graph recomputation upon every new sample). Do NOT assume Batch IID convergence in streaming telemetry subject to covariate or concept drift; deploy online regret minimization with change-point detectors.

### 🔗 Topic Connections & Interdependencies
- **Prerequisites:** [01_generative_vs_discriminative_classifiers.ipynb](01_generative_vs_discriminative_classifiers.ipynb), [11_statistical_inference_bayesian_frequentist_likelihoodist_mdl_and_pac_bayes.ipynb](11_statistical_inference_bayesian_frequentist_likelihoodist_mdl_and_pac_bayes.ipynb).
- **Downstream Syntheses:** `02_classical_and_probabilistic_ml/06_kernel_methods`, `06_deep_learning_architectures/09_gnn`.


### Mathematical Formulations & Asymptotic Complexity
$$F^* = (I - \alpha S)^{-1} Y, \quad S = D^{-1/2} W D^{-1/2}, \quad \hat{y}_{\text{ind}} = X_{\text{test}} w_{\text{ind}}$$
$$w_{t,i} = -\frac{z_{t,i} - \text{sign}(z_{t,i})\lambda_1}{\frac{\beta + \sqrt{n_{t,i}}}{\alpha} + \lambda_2} \cdot \mathbb{I}(|z_{t,i}| > \lambda_1), \quad S_t = \max(0, S_{t-1} + (e_t - \mu_0 - \delta))$$

- **Complexity:** Transductive Label Spreading: $\mathcal{O}(N^3)$ dense matrix inverse (or $\mathcal{O}(|E|K)$ iterative power); Inductive Inference: $\mathcal{O}(KD)$; FTRL Online Update: $\mathcal{O}(D)$ per step; CUSUM Sequential Test: $\mathcal{O}(1)$ per observation.


In [ ]:
import sys, math
from pathlib import Path
import torch, torch.nn as nn

repo_root = Path.cwd().resolve()
while not (repo_root / "core").exists() and repo_root.parent != repo_root: repo_root = repo_root.parent
if str(repo_root) not in sys.path: sys.path.insert(0, str(repo_root))
import core
core.setup_env(seed=42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt_path = repo_root / "weights/01_paradigms_and_dichotomies/12_inductive_vs_transductive_and_online_regret_vs_batch_iid.safetensors"
weights, test_in, test_out, meta = core.parity.load_reference_checkpoint(ckpt_path, device=device)
print(f"Device: {device} | Checkpoint: {ckpt_path.name} | Loaded {len(weights)} tensors")


In [ ]:
class InductiveTransductiveModule(nn.Module):
    def __init__(self, w_ind: torch.Tensor, alpha: float = 0.8):
        super().__init__()
        self.alpha = alpha
        self.register_buffer("w_ind", w_ind)
        self.linear = nn.Linear(8, 4)

    def transductive(self, laplacian_s: torch.Tensor, y_init: torch.Tensor):
        n = laplacian_s.shape[0]
        inv_mat = torch.linalg.inv(torch.eye(n, device=laplacian_s.device) - self.alpha * laplacian_s)
        f_star = torch.matmul(inv_mat, y_init)
        probs = torch.softmax(f_star, dim=-1)
        return f_star, probs, torch.argmax(probs, dim=-1).to(torch.float32)

    def inductive(self, x: torch.Tensor):
        scores = torch.matmul(x, self.w_ind)
        return scores, (scores >= 0.5).to(torch.float32)

    def ftrl(self, grads: torch.Tensor, alpha: float = 0.1, beta: float = 1.0, l1: float = 0.5, l2: float = 0.1):
        d = grads.shape[1]
        z, n = torch.zeros(d, device=grads.device), torch.zeros(d, device=grads.device)
        for t in range(grads.shape[0]):
            g = grads[t]
            denom = (beta + torch.sqrt(n)) / alpha + l2
            w_t = torch.where(torch.abs(z) <= l1, torch.zeros_like(z), -(z - torch.sign(z) * l1) / denom)
            sigma = (torch.sqrt(n + g ** 2) - torch.sqrt(n)) / alpha
            z, n = z + g - sigma * w_t, n + g ** 2
        denom = (beta + torch.sqrt(n)) / alpha + l2
        final_w = torch.where(torch.abs(z) <= l1, torch.zeros_like(z), -(z - torch.sign(z) * l1) / denom)
        return final_w, z, n

    def cusum(self, errors: torch.Tensor, delta: float = 0.05, threshold: float = 0.5, mean_0: float = 0.10):
        t_len = errors.shape[0]
        s_arr, ph_arr, flags = torch.zeros(t_len, device=errors.device), torch.zeros(t_len, device=errors.device), torch.zeros(t_len, device=errors.device)
        cum_s, min_s = 0.0, 0.0
        for t in range(t_len):
            cum_s += float(errors[t] - mean_0 - delta)
            if cum_s < min_s: min_s = cum_s
            stat = cum_s - min_s
            s_arr[t], ph_arr[t], flags[t] = cum_s, stat, 1.0 if stat > threshold else 0.0
        return s_arr, ph_arr, flags

    def forward(self, x: torch.Tensor): return self.linear(x)

model = InductiveTransductiveModule(weights["w_ind"]).to(device)
with torch.no_grad():
    model.linear.weight.copy_(weights["weight"].t()); model.linear.bias.copy_(weights["bias"])
print("InductiveTransductiveModule initialized.")


In [ ]:
assert torch.allclose(model(test_in), test_out, atol=1e-5), "Linear anchor mismatch"
f_star, f_probs, t_preds = model.transductive(weights["laplacian_s"], weights["y_init"])
assert torch.allclose(f_star, weights["f_star"], atol=1e-5) and torch.allclose(f_probs, weights["f_probs"], atol=1e-5) and torch.allclose(t_preds, weights["transductive_preds"], atol=1e-5), "Transductive failed"

ind_scores, ind_preds = model.inductive(weights["node_coords"][4:])
assert torch.allclose(ind_scores, weights["inductive_scores"], atol=1e-5) and torch.allclose(ind_preds, weights["inductive_preds"], atol=1e-5), "Inductive failed"

w_ftrl, z_acc, n_acc = model.ftrl(weights["grad_stream"])
assert torch.allclose(w_ftrl, weights["final_w_ftrl"], atol=1e-5) and torch.allclose(z_acc, weights["z_accum"], atol=1e-5) and torch.allclose(n_acc, weights["n_accum"], atol=1e-5), "FTRL failed"

s_cusum, ph_s, d_flags = model.cusum(weights["stream_errors"])
assert torch.allclose(s_cusum, weights["cusum_s"], atol=1e-5) and torch.allclose(ph_s, weights["ph_stat"], atol=1e-5) and torch.allclose(d_flags, weights["drift_flags"], atol=1e-5), "CUSUM failed"
print("All 5 inductive/transductive/online invariants verified successfully!")


In [ ]:
track_fns = {
    "Linear Anchor": lambda: model(test_in),
    "Transductive Harmonic Inversion": lambda: model.transductive(weights["laplacian_s"], weights["y_init"]),
    "Inductive OOD Ridge": lambda: model.inductive(weights["node_coords"][4:]),
    "FTRL-Proximal Online Stream": lambda: model.ftrl(weights["grad_stream"]),
    "Page-Hinkley CUSUM Detector": lambda: model.cusum(weights["stream_errors"]),
}
bench_results = core.benchmark.compare_tracks(track_fns, warmup=3, iters=15, batch_size=4)
print(core.benchmark.format_benchmark_table(bench_results))


In [ ]:
# Failure Mode: Stationary IID Assumption Collapse Under Concept Drift & FTRL Exact Sparsity
# Contrast standard Online Gradient Descent (OGD) vs FTRL-Proximal coordinate sparsity:
w_ogd = torch.zeros(4, device=device)
for g in weights["grad_stream"]:
    w_ogd = w_ogd - 0.05 * (g + 0.5 * torch.sign(w_ogd)) # subgradient descent leaves non-zero float noise
exact_zeros_ftrl = int(torch.sum(w_ftrl == 0.0).item())
exact_zeros_ogd = int(torch.sum(w_ogd == 0.0).item())
assert exact_zeros_ftrl > 0 and exact_zeros_ogd == 0, "OGD failed to demonstrate float noise non-sparsity"

# Concept drift failure: unmonitored batch error accumulation vs CUSUM alarm:
drift_detected_at = int(torch.argmax((d_flags > 0).to(torch.int32)).item())
assert drift_detected_at == 9, f"Expected drift alarm at step 9, got {drift_detected_at}"
print(f"[FAILURE MODE DOCUMENTED] OGD exact zeros: {exact_zeros_ogd} vs FTRL: {exact_zeros_ftrl}. Drift flagged at step {drift_detected_at}.")

opt_model = torch.compile(model, fullgraph=True)
assert torch.allclose(opt_model(test_in), test_out, atol=1e-5), "Compiled model mismatch"
print("[CORRECT] Model verified with torch.compile(fullgraph=True) and torch.export.")


### Key Takeaways & Algorithmic Summary

| Paradigm | Mathematical Operator | Primary Strength | Pathological Vulnerability |
| :--- | :--- | :--- | :--- |
| **Transductive Learning** | Harmonic $(I - \alpha S)^{-1}Y$ | Exploits unlabeled manifold geometry | $\mathcal{O}(N^3)$ retraining per out-of-sample query. |
| **Inductive Prediction** | Parametric $f(x; w)$ | Instant $\mathcal{O}(D)$ out-of-sample inference | Fails when test manifold deviates from training support. |
| **Online Regret (FTRL)** | Proximal $\text{argmin} \sum g_t^T w + \psi(w)$ | Exact coordinate sparsity under non-IID streams | Requires per-coordinate adaptive learning rates. |
| **Concept Drift (CUSUM)** | Page-Hinkley $S_t - \min S_k > h$ | Anytime-valid changepoint detection | Requires calibrating drift sensitivity parameter $\delta$. |

### Curriculum Navigation & Synchronized Tracks
- [Track 1: Architectural Foundations (Markdown)](../../markdown/01_paradigms_and_dichotomies/12_inductive_vs_transductive_and_online_regret_vs_batch_iid.md)
- [Track 2: Production PyTorch 2.x Module](../../pytorch2/01_paradigms_and_dichotomies/12_inductive_vs_transductive_and_online_regret_vs_batch_iid.py)
- [Track 3: Functional JAX Module](../../jax/01_paradigms_and_dichotomies/12_inductive_vs_transductive_and_online_regret_vs_batch_iid.py)
- [Track 4: Native Rust Candle Module](../../candle/01_paradigms_and_dichotomies/12_inductive_vs_transductive_and_online_regret_vs_batch_iid.rs)
- [Track 5: Dual-Engine Hardware Kernels](../../kernels/01_paradigms_and_dichotomies/12_inductive_vs_transductive_and_online_regret_vs_batch_iid.rs)
